In [ ]:
from datasets import Dataset

from tklearn.nn.models.kbert.tokenizer import KnowledgeBaseTokenizer
from tklearn.plotting.matrix import plot_dot_matrix
from tklearn.plotting.token_tree import TokenTree

In [ ]:
# ~35 seconds to run
tokenizer = KnowledgeBaseTokenizer({
    "model_name_or_path": "bert-base-uncased",
})

In [ ]:
# dataset = load_dataset("ag_news", split="train[:1%]")
# dataset[0]
# # https://huggingface.co/datasets/google/civil_comments
# dataset = load_dataset("google/civil_comments", split="train[:2000]")

In [ ]:
tokenizer.k = 2

In [ ]:
dataset = Dataset.from_dict({
    "text": [
        "Tim Cook is the CEO of Apple.",
        # "The Beauty and the Beast is a fantastic tale.",
        # example where KB is useful in online behavior analysis - Disaster Tweets
        # "Floods in India have caused massive damage to infrastructure and homes."
        # "The lab synthesized a green fluorescent protein derivative."
    ]
})

In [ ]:
# dataset = dataset.map(tokenizer, batched=True, load_from_cache_file=False)
dataset = dataset.map(tokenizer, batched=True)

In [ ]:
dataset.set_format(type="torch")

In [ ]:
dataset[0].keys()

In [ ]:
example = dataset[0]

tree = TokenTree.loads(example["token_tree"])
tree.build()

M = example["visibility_matrix"]

plot_dot_matrix(M)

tree.graphviz()

In [ ]:
from sentence_transformers import CrossEncoder

# 1. Load an STS (Semantic Similarity) Model
# This model is optimized to compare meaning, not search relevance.
# model = CrossEncoder("cross-encoder/stsb-distilroberta-base")
# ms-marco-MiniLM-L-6-v2
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def analyze_context_definition_matching(
    text: str,
    target_word: str,
    definitions: list[str],
):
    # 2. Format the pairs
    # Strategy: Compare the [Context] directly with the [Definition]
    # We prefix the definition with the word to help the model link them.
    pairs = []
    for defn in definitions:
        # Pair: (Context Sentence,  Target Word + Definition)
        pairs.append([
            f"Meaning of {target_word} in {text}",
            defn,
        ])

    # 3. Predict (Scores will be 0 to 1)
    scores = model.predict(pairs, convert_to_numpy=True).tolist()

    # 4. Rank
    results = list(zip(definitions, scores))
    results.sort(key=lambda x: x[1], reverse=True)

    return results

In [ ]:
dataset[0].keys()

In [ ]:
import pandas as pd

example = dataset[0]

text = example["original_text"]
form = example["triples"][5]["mention.text"]

sense_ids = set()

for word, _ in tokenizer.knowledge_base.lexicon.get(form):
    sense_ids = sense_ids.union(tokenizer.knowledge_base.senses[word])

sense_ids = list(sense_ids)
definitions = []
for sense_id in sense_ids:
    definitions.append(tokenizer.knowledge_base.idx2gloss.get(sense_id))


print(f"Analyzing word '{form}' in context: {text}")
pd.DataFrame(
    analyze_context_definition_matching(
        text=text,
        target_word=form,
        definitions=definitions,
    )
).iloc[:5][0].tolist()

In [ ]:
# # read https://raw.githubusercontent.com/hate-alert/HateXplain/refs/heads/master/Data/dataset.json
# df = pd.read_json(
#     "https://raw.githubusercontent.com/hate-alert/HateXplain/refs/heads/master/Data/dataset.json",
#     orient="index",
# )
# df = df.assign(
#     text=df["post_tokens"].apply(lambda tokens: " ".join(tokens)),
#     annotators=df["annotators"]
#     .apply(
#         lambda annots: pd.Series([a["label"] for a in annots]).value_counts()
#     )
#     .fillna(0)
#     .idxmax(axis=1),
# )
# df

In [ ]:
dataset["original_text"][0]

In [ ]:
example["original_text"]

In [ ]:
list(
    tokenizer.knowledge_base.extract_mentions(
        "The misogynists were angry about the new policies."
    )
)